<img src = "https://drive.google.com/uc?export=view&id=1Hh_G3M13P9xSNgSiQ-WnALg93XwK_hG8" alt = "Encabezado MLDS" width = "100%">  </img>

# **Tarea 5 - *Modelos generativos***
---

En esta tarea utilizarás Transformers para convertir texto natural en representaciones mediante emojis. El ejercicio permite comprender cómo los modelos generativos capturan la semántica del texto y su contexto emocional.

> **Importante: Recomendamos utilizar GPU para la ejecución de este notebook, ya que puede tomar mucho tiempo la ejeución de algunos casos de prueba en caso de que no se utilice. Tenga en cuenta que, con el propósito de guardar recursos, los primeros puntos se pueden solucionar sin utilizarlo**

**Descripción del problema**
---
La representación de texto mediante emojis trasciende la traducción directa palabra-a-símbolo. Requiere que el modelo:
- Comprenda el contexto semántico del texto de entrada
- Identifique conceptos clave y emociones implícitas
- Genere secuencias coherentes de emojis que preserven el significado original
- Mantenga la fluidez en la representación visual

# 📘 Tarea 5 · Versión explicada

Esta es una copia del notebook resuelto con **una explicación antes de cada celda de código**. El código y las salidas son los de la ejecución real que obtuvo **100 % en UNCode**, con las 11 pruebas en 100. El notebook original, sin explicaciones, es `v2_M5U5_Tarea_5_Resuelto.ipynb`.

**Cómo leerlo:** las celdas que empiezan con 📘 explican el código que viene justo después; las que empiezan con 🔎 interpretan la salida que quedó arriba. Las celdas originales del enunciado se conservan tal cual.

| | |
|---|---|
| **Problema** | Convertir frases en inglés en secuencias de emojis: un problema de secuencia a secuencia, como la traducción automática |
| **Modelo** | Transformer **codificador-decodificador**. El codificador lee el inglés; el decodificador escribe los emojis uno a uno y consulta al codificador con **atención cruzada** |
| **Datos** | Unas 10.000 parejas de frase y emojis, en su mayoría del Parlamento Europeo: 6.605 de entrenamiento y 3.129 de prueba |
| **Resultado** | La pérdida bajó de 8,03 a 3,48; la mejor validación fue 3,11 en la época 19 |
| **Correcciones clave** | Emojis **sin separar**, **nombres exactos** de los atributos de las capas y `dropout=rate` en la atención |

**Flujo del notebook**

```
JSON frase → emojis
   → Punto 1 preprocess        limpieza y [BOS] … [EOS]
   → Punto 2 vectorize         enteros de longitud 26
   → Punto 3 codificador       (lote, 26)            → (lote, 26, 128)
   → Punto 4 decodificador     (lote, 25) + contexto → (lote, 25, 5000)
   → Punto 5 get_model         une ambas capas, 2.345.416 parámetros
   → Punto 6 train_model       Adam + causal_loss, 40 épocas, desfase de un token
   → Generación token a token: [BOS] → … → [EOS]
```

**Dos ideas que atraviesan todo el notebook**
- **Por qué 26 y 25.** El vectorizador produce 26 posiciones. El codificador usa las 26. Para entrenar, a los emojis se les quita un token de cada lado: la entrada del decodificador es `[:, :-1]` y el objetivo, `[:, 1:]`. Ambos miden 25.
- **Teacher forcing.** En cada posición el decodificador ve el token correcto actual y debe predecir el siguiente. La máscara causal le impide mirar hacia adelante.


### 📘 Explicación

Instala y descarga el calificador de UNCode.

- `%%capture` oculta la salida de la celda y el signo `!` ejecuta un comando de terminal.
- `rlxcrypt` descifra el módulo `session.pye`, que viene cifrado para que no se pueda leer la solución de referencia.

In [39]:
%%capture
!pip install rlxcrypt
!wget --no-cache -O session.pye -q https://raw.githubusercontent.com/JuezUN/INGInious/master/external%20libs/session.pye

### 📘 Explicación

Crea el objeto `grader`, vinculado al curso y a esta tarea. Pide el usuario y la contraseña de UNCode; la contraseña no queda guardada. Todas las celdas `grader.` dependen de esta.

In [40]:
import rlxcrypt
import session

grader = session.LoginSequence("DL-GroupMLDS-5-2026-2@8c487bd7-4c95-4b75-88d5-9e7093aae729")

Please enter your UNCode username: nremolina
Please enter your password: ··········


> Al finalizar, podrás revisar en UNCode tus submissions en la [página de la tarea.](https://uncode.unal.edu.co/course/DL-GroupMLDS-5-2026-2/8c487bd7-4c95-4b75-88d5-9e7093aae729)

### 📘 Explicación

Importa las librerías y fija las semillas.

- `re` sirve para las expresiones regulares de la limpieza y `json`, para leer el archivo de datos.
- **Las tres semillas** fijan el azar de Python, Keras y NumPy. El calificador compara **valores numéricos exactos** en varias pruebas, así que sin semillas los pesos iniciales nunca coincidirían con los de la referencia.

In [41]:
import keras
import numpy as np
import matplotlib.pyplot as plt
import os, random
import pandas as pd
import re
import json


%matplotlib inline
plt.style.use("ggplot")
# Seleccionamos las semillas para efectos de reproducibilidad
random.seed(0)
keras.utils.set_random_seed(0)
np.random.seed(0)

### 📘 Explicación

Imprime las versiones del entorno. Es solo informativa.

In [42]:
# Versiones de las librerías usadas.
!python --version
print('NumPy', np.__version__)
print('keras', keras.__version__)

Python 3.13.16
NumPy 2.4.6
keras 3.13.2


#### 🔎 Lectura de la salida

Las versiones son más nuevas que las del enunciado, que usó Python 3.11.12, NumPy 2.0.2 y Keras 3.8.0, y aun así todas las pruebas pasaron.

Esta actividad se realizó con las siguientes versiones:
* Python 3.11.12
* NumPy 2.0.2
* keras 3.8.0

## **Cargar Datos**
---

Vamos cargar nuestro conjunto de datos. Este posee dos columnas: la primera contiene 10,000 textos en inglés  y en la segunda su representación por medio de emojis. Si bien esta puede parecer una tarea de traducción simple, el uso de emojis como concepto le exige al modelo la capacidad de abstraer patrones complejos.

### 📘 Explicación

Descarga `emoji_translations.json` de Google Drive con `gdown`, una herramienta que baja archivos a partir de su identificador.

In [43]:
!pip install gdown
!gdown https://drive.google.com/uc?id=1HjGoRLLnLdEWjB1peXGFQXNGvbPRaAlj

Downloading...
From: https://drive.google.com/uc?id=1HjGoRLLnLdEWjB1peXGFQXNGvbPRaAlj
To: /content/emoji_translations.json
100% 1.09M/1.09M [00:00<00:00, 12.0MB/s]


### 📘 Explicación

Lee el archivo y lo divide en entrenamiento y prueba.

- `encoding='utf-8'` es imprescindible: los emojis son caracteres Unicode de varios bytes.
- `json.load` devuelve un diccionario `{frase: emojis}`; `.items()` lo convierte en pares y el DataFrame los pone en las columnas `ingles` y `emojis`.
- `dropna()` quita filas vacías, pero **no renumera el índice**.

> ⚠️ **La división no es exactamente 70 / 30.** `.loc` trabaja con etiquetas e **incluye** el extremo final. Como `dropna()` dejó huecos en el índice, la división real es 67,9 % y 32,1 %, y la fila de la frontera queda en los dos conjuntos. No se corrigió porque viene en la plantilla y el calificador espera exactamente 6.605 filas de entrenamiento.

In [44]:
with open("/content/emoji_translations.json", 'r', encoding='utf-8') as f:
    data_dict = json.load(f)

data = pd.DataFrame(data_dict.items(), columns=['ingles', 'emojis'])
data = data.dropna()

train_texts = data.loc[:int(len(data)*0.7)]
test_texts = data.loc[int(len(data)*0.7):]

print(f'Numero de textos de entrenamiento: {len(train_texts)}')
print(f'Numero de textos de prueba: {len(test_texts)}')

Numero de textos de entrenamiento: 6605
Numero de textos de prueba: 3129


veamos algunos ejemplos:

### 📘 Explicación

Muestra las cinco primeras filas para ver el formato de los datos.

In [45]:
train_texts.head()

,ingles,emojis
0,the ratification and implementation of the upd...,✅🗳️
1,sustainable agriculture and biogas review of e...,🌱♻️🇪🇺
2,before the vote on paragraph,🗳️📄
3,plus we agreed to delete point a,❌
4,parliament agreed to accept the oral amendment,✅💬


#### 🔎 Lectura de la salida

Las frases son en su mayoría intervenciones del Parlamento Europeo, y los emojis resumen la idea: ✅ para aprobar, ❌ para eliminar, 🗳️ para votar. La fila 0, *ratification … vote* → ✅🗳️, sirve como ejemplo en el resto del notebook.

# **1. Preprocesamiento**
---
En este primer punto deberá implementar una función de prerpocesamiento de texto adaptada al problema presentado. Para eso deberá:


**Para la columna de texto:**

* Convertir todo el texto a minúsculas
* Eliminar caracteres especiales (mantener solo letras y espacios)
* Eliminar espacios repetidos
* Añadir los tokens de inicio `[BOS]` y final `[EOS]`

**Para la columna de emojis:**

* Añadir los tokens de inicio `[BOS]` y final `[EOS]`

**Entradas**:

* **`data`**: `pd.DataFrame`,  DataFrame que contiene las columnas 'texto' y 'emojis'

**Salida**:

* **`text_out`**: `pd.DataFrame`, DataFrame con el texto preprocesado según las reglas anteriores

### 📘 Explicación

**Punto 1.** Limpia el inglés y marca inicio y fin en ambas columnas.

**Línea por línea**
1. `text.lower()`: «The» y «the» pasan a ser el mismo token, lo que reduce el vocabulario.
2. `re.sub(r'[^a-z\s]', '', text)`: `[^a-z\s]` significa «cualquier carácter que **no** sea letra minúscula ni espacio» y se borra. Desaparecen cifras, puntuación, paréntesis y apóstrofes. Va **después** de `lower()` para no borrar las mayúsculas.
3. `re.sub(r'\s+', ' ', text).strip()`: cambia los espacios repetidos por uno solo y quita los de los extremos. Hace falta porque al borrar símbolos quedan espacios dobles.
4. `'[BOS] ' + text + ' [EOS]'`: añade las marcas de inicio y fin, separadas por espacios para que el vectorizador las tome como tokens propios.
5. `preprocess_emoji` solo añade las marcas: los emojis **no se tocan**.

**Qué le pasa a cada texto**

| Entrada | Salida |
|---|---|
| `I'm so happy (today)` | `[BOS] im so happy today [EOS]` |
| `Let's eat some pizza!!!` | `[BOS] lets eat some pizza [EOS]` |
| `✅🗳️` | `[BOS] ✅🗳️ [EOS]` |

> ⚠️ **Primera versión: 80 en las pruebas 1_1 y 1_2.** La salida esperada del enunciado muestra los emojis separados por espacios, `✅ 🗳 ️`, y la primera versión los separaba carácter por carácter. La instrucción escrita dice solo «añadir los tokens». Se probaron cuatro variantes contra el calificador y solo la que deja los emojis intactos dio 100.
>
> La salida impresa tiene además una pista: el espacio suelto antes del último símbolo. Ese carácter invisible es el selector de variación U+FE0F, que forma parte de 🗳️. Separar con `' '.join(emoji)` lo corta por puntos de código Unicode.

In [46]:
# FUNCIÓN CALIFICADA preprocess
def preprocess(data):
    def preprocess_text(text):
        text = text.lower()                         # minúsculas
        text = re.sub(r'[^a-z\s]', '', text)        # solo letras y espacios
        text = re.sub(r'\s+', ' ', text).strip()    # espacios repetidos
        return '[BOS] ' + text + ' [EOS]'
    def preprocess_emoji(emoji):
        return '[BOS] ' + emoji + ' [EOS]'

    text_out = data.copy()
    text_out['ingles'] = text_out['ingles'].map(preprocess_text)
    text_out['emojis'] = text_out['emojis'].map(preprocess_emoji)
    return text_out

### 📘 Explicación

Aplica `preprocess` a entrenamiento y prueba e imprime la fila 0 de ambas columnas.

In [47]:
# TEST CELL
train_texts_preprocess = preprocess(train_texts.copy())
test_texts_preprocess = preprocess(test_texts.copy())


print('Texto en inglés')
print(train_texts_preprocess.iloc[0]['ingles'])
print('Emojis')
print(train_texts_preprocess.iloc[0]['emojis'])

Texto en inglés
[BOS] the ratification and implementation of the updated ilo conventions vote [EOS]
Emojis
[BOS] ✅🗳️ [EOS]


#### 🔎 Lectura de la salida

«The» pasó a «the», «ILO» a «ilo» y la frase quedó delimitada. Los emojis conservan su forma original, **sin separar**: así lo exige el calificador, aunque la salida esperada del enunciado los muestre separados.

**Salida esperada:**
```
Texto en ingles
[BOS] the ratification and implementation of the updated ilo conventions vote [EOS]
Emojis
[BOS] ✅ 🗳 ️ [EOS]
```




## **Evaluar código**

### 📘 Explicación

**Pruebas del punto 1.** El calificador aplica `preprocess` a sus propios textos, con mayúsculas, puntuación y paréntesis, y compara el DataFrame resultante con el esperado.

In [48]:
grader.run_test("Test 1_1", globals())

Test 1_1


In [49]:
grader.run_test("Test 1_2", globals())

Test 1_2


# **2. Vectorización**
----
Cree y ajuste una capa de vectorización para cada tipo de secuencia (texto y emojis).

**Entrada**:
* **`X`**:  `pd.DataFrame`, DataFrame que contiene el texto normalizado en `ingles` y `emoji`.
* **`seq_len`**: Tamaño máximo de la secuencia.
* **`vocab_size`**: Tamaño máximo del vocabulario.

**Salida**:
* **`en_vectorizer`**:  `keras.layer`, Vectorizador ajustado para el idioma inglés.

* **`em_vectorizer`**:  `keras.layer`, Vectorizador ajustado para el idioma emojis.


> **Nota**:Tenga presente que en el caso de la columna emojis su función solo debera añadir los tokens de incio y de parada, mientras que el texto debe ser preprocesado.

### 📘 Explicación

**Punto 2.** Construye un vectorizador para el inglés y otro para los emojis.

| Parámetro | Valor | Por qué |
|---|---|---|
| `max_tokens` | 5.000 | Solo los 5.000 tokens más frecuentes tienen id propio; el resto se vuelve `[UNK]` |
| `output_mode` | `'int'` | Cada token se convierte en su id entero |
| `output_sequence_length` | `seq_len + 1` = 26 | Rellena o recorta a 26. El +1 es por el desfase del entrenamiento: al quitar un token de cada lado quedan 25 |
| `standardize` | `None` | Por defecto la capa quita la puntuación y borraría los corchetes de `[BOS]` y `[EOS]`. El texto ya viene limpio |
| `.adapt(...)` | Solo entrenamiento | Construye el vocabulario ordenado por frecuencia. Si viera la prueba, la evaluación quedaría contaminada |

Los cuatro primeros ids son fijos: **0** relleno, **1** `[UNK]`, **2** `[EOS]` y **3** `[BOS]`. Por eso la generación del final arranca con 3 y se detiene en 2.

In [50]:
# FUNCIÓN CALIFICADA vectorize
def vectorize(X, seq_len, vocab_size):
    en_vectorizer = keras.layers.TextVectorization(
        max_tokens=vocab_size, output_mode='int',
        output_sequence_length=seq_len + 1, standardize=None)
    em_vectorizer = keras.layers.TextVectorization(
        max_tokens=vocab_size, output_mode='int',
        output_sequence_length=seq_len + 1, standardize=None)
    en_vectorizer.adapt(X['ingles'].values)
    em_vectorizer.adapt(X['emojis'].values)
    return en_vectorizer, em_vectorizer

### 📘 Explicación

Crea los vectorizadores con `seq_len = 25` y `vocab_size = 5000` y transforma los cuatro conjuntos. `.numpy()` convierte los tensores en arreglos de NumPy.

In [51]:
# TEST_CELL
en_vectorizer, emoji_vectorizer = vectorize(train_texts_preprocess, 25, 5000)
train_en_vec, train_emoji_vec = en_vectorizer(train_texts_preprocess['ingles']).numpy(), emoji_vectorizer(train_texts_preprocess['emojis']).numpy()
test_en_vec, test_emoji_vec = en_vectorizer(test_texts_preprocess['ingles']).numpy(), emoji_vectorizer(test_texts_preprocess['emojis']).numpy()

print(f'Tamaño de las entradas: {train_en_vec.shape}')
print(f'Tamaño de las salidas: {train_emoji_vec.shape}')

Tamaño de las entradas: (6605, 26)
Tamaño de las salidas: (6605, 26)


#### 🔎 Lectura de la salida

Matrices de 6.605 filas por 26 columnas: una fila por frase y una columna por posición.

> **Consecuencia de no separar los emojis.** Los primeros tokens del vocabulario de emojis son `'' · [UNK] · [EOS] · [BOS] · 🤔❓ · ✅👍 · ❓🤔 · ✅ · 🤔 · 🤔📝❓`. Como la capa separa por espacios, **cada cadena completa de emojis es un solo token**: `🤔❓` y `❓🤔` son tokens distintos y sin relación. Toda fila de emojis queda como `[3, id, 2, 0, 0, …]`, y el modelo debe elegir la cadena exacta entre miles. Eso hace la tarea mucho más difícil.

**Salida esperada**:
```
Tamaño de las entradas: (6605, 26)
Tamaño de las salidas: (6605, 26)

```

## **Evaluar código**

### 📘 Explicación

**Pruebas del punto 2.** Las dos dieron 100 en el primer intento.

In [52]:
grader.run_test("Test 2_1", globals())

Test 2_1


In [53]:
grader.run_test("Test 2_2", globals())

Test 2_2


# **3. Codificador**
----
Implemente una capa personalizada de Keras que represente el codificador de un Transformer.


**Entrada**:
* **`vocab_size`**: `int`, Tamaño del vocabulario de entrada
* **`maxlen`**: `int`, Longitud máxima de la secuencia de entrada
* **`embed_dim`**: `int`, Dimensión del espacio de embedding para cada token
* **`num_heads`**: `int`, Número de cabezas en el mecanismo de auto-atención
* **`ff_dim`**: `int`, Dimensión de la capa oculta en la red feed-forward
* **`rate`**: `float`, Tasa de dropout para regularización

**Salida**:
* **`layer`**:  `keras.layer.layer`, Capa personalizada de _Keras_.

### 📘 Explicación

**Punto 3.** El codificador: lee la frase en inglés y produce, para cada palabra, un vector que ya incorpora información de toda la frase.

**Piezas que se declaran en `__init__`**

| Atributo | Qué es |
|---|---|
| `token_emb` | Tabla de 5.000 × 128: un vector por palabra |
| `pos_emb` | Tabla de 26 × 128: un vector por posición. Es `maxlen + 1` porque la entrada mide 26 |
| `self_att` | Atención multicabeza, 2 cabezas, `key_dim=embed_dim` y **`dropout=rate`** |
| `self_layernorm`, `self_dropout` | Normalización y dropout del bloque de atención |
| `ffn` | Red de dos capas: 128 → 32 con ReLU → 128 |
| `dropout`, `layernorm` | Normalización y dropout del bloque *feed-forward* |

**Lo que hace `call`, con un lote de 10 frases**

| Paso | Forma |
|---|---|
| `inputs` | `(10, 26)` |
| `token_emb(inputs) + pos_emb(positions)`: qué palabra es más dónde está | `(10, 26, 128)` |
| `self_att(x, x)`: **auto-atención**, cada posición mezcla información de las 26 | `(10, 26, 128)` |
| `self_layernorm(x + attn)`: suma residual y normalización | `(10, 26, 128)` |
| `ffn(x)` | `(10, 26, 32)` → `(10, 26, 128)` |
| `layernorm(x + f)`: el contexto que leerá el decodificador | `(10, 26, 128)` |

> ⚠️ **Primera versión: 0 y 10.** El código era correcto en lo matemático, pero usaba nombres propios como `att`, `norm1` o `drop1`. El error *"'encoder' object has no attribute 'self_att'"* reveló que la prueba **inspecciona los atributos por su nombre**. Con una celda temporal que registraba cada atributo leído se obtuvieron los nombres exactos que aparecen arriba. Después, la prueba exigió también `dropout=rate` en la atención: *"Se esperaba 0.1, pero se obtuvo 0.0"*.

In [54]:
# FUNCIÓN CALIFICADA get_EncoderLayer
def get_EncoderLayer(vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate):

    class encoder(keras.layers.Layer):
        def __init__(self, vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate=0.1):
            super().__init__()
            self.token_emb = keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
            self.pos_emb = keras.layers.Embedding(input_dim=maxlen + 1, output_dim=embed_dim)
            self.self_att = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=rate)
            self.self_layernorm = keras.layers.LayerNormalization()
            self.self_dropout = keras.layers.Dropout(rate)
            self.ffn = keras.Sequential([keras.layers.Dense(ff_dim, activation="relu"),
                                         keras.layers.Dense(embed_dim)])
            self.dropout = keras.layers.Dropout(rate)
            self.layernorm = keras.layers.LayerNormalization()

        def call(self, inputs):
            positions = keras.ops.arange(start=0, stop=keras.ops.shape(inputs)[-1], step=1)
            x = self.token_emb(inputs) + self.pos_emb(positions)
            attn = self.self_att(x, x)
            attn = self.self_dropout(attn)
            x = self.self_layernorm(x + attn)
            f = self.ffn(x)
            f = self.dropout(f)
            return self.layernorm(x + f)

    layer = encoder(vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate)
    return layer

### 📘 Explicación

Codifica las 10 primeras frases de entrenamiento.

In [55]:
# TEST_CELL
encoder_layer = get_EncoderLayer(5000, 25, 128, 2, 32, .1)
encoded_x = encoder_layer(train_en_vec[:10])
print(f'Codificación de sequencias de entrenamiento: {encoded_x.shape}')

Codificación de sequencias de entrenamiento: (10, 26, 128)


#### 🔎 Lectura de la salida

Para cada frase, el codificador entrega 26 vectores de 128 valores, uno por posición, y cada uno contiene información de toda la frase. Es el «contexto» que consultará el decodificador.

**Salida esperada**
```
Codificación de sequencias de entrenamiento: (10, 26, 128)
```

## **Evaluar código**

### 📘 Explicación

**Pruebas del punto 3.** La 3_1 revisa la estructura de la capa: nombres, tipos y parámetros de cada atributo. La 3_2 compara los valores numéricos de la salida con la referencia.

In [56]:
grader.run_test("Test 3_1", globals())

Test 3_1


In [57]:
grader.run_test("Test 3_2", globals())

Test 3_2


# **4. Decodificador**
----
Implemente una capa personalizada de Keras que represente el decodificador de un Transformer.

**Entrada**:

* **`vocab_size`**: `int`, Tamaño del vocabulario de salida
* **`maxlen`**: `int`, Longitud máxima de la secuencia de salida  
* **`embed_dim`**: `int`, Dimensión del espacio de embedding para cada token
* **`num_heads`**: `int`, Número de cabezas en los mecanismos de atención
* **`ff_dim`**: `int`, Dimensión de la capa oculta en la red feed-forward
* **`rate`**: `float`, Tasa de dropout para regularización

**Salida**:
* **`layer`**:  `keras.layer.layer`, Capa personalizada de _Keras_.

### 📘 Explicación

**Punto 4.** El decodificador: escribe los emojis mirando lo que ya escribió y consultando la frase en inglés.

**Lo que cambia respecto al codificador**

| Elemento | Detalle | Por qué |
|---|---|---|
| `pos_emb` de 25 | `input_dim=maxlen`, sin el +1 | La entrada son los emojis sin su último token |
| **Máscara causal** | `self_att(x, x, use_causal_mask=True)` | La posición *i* solo puede mirar las posiciones 0 a *i*. Sin ella, al entrenar vería el token que debe predecir, y al generar fallaría porque ese futuro no existe |
| **Atención cruzada** | `cross_att(x, context_tokens)` | Las consultas salen de los emojis; las claves y los valores, del codificador. Es la **única vía** por la que el inglés influye en la salida |
| `cross_layernorm`, `cross_dropout` | Normalización y dropout del bloque cruzado | Mismo patrón residual |
| `out_dropout`, `out_layer` | `Dense(5000, softmax)` | Una probabilidad por token del vocabulario de emojis. Es *softmax* porque `causal_loss` usa `from_logits=False` |

**La máscara causal con cuatro posiciones:** ✓ significa que puede mirar.

| | `[BOS]` | e₁ | e₂ | e₃ |
|---|:-:|:-:|:-:|:-:|
| `[BOS]` | ✓ | | | |
| e₁ | ✓ | ✓ | | |
| e₂ | ✓ | ✓ | ✓ | |
| e₃ | ✓ | ✓ | ✓ | ✓ |

**Flujo de `call`:**
1. Embeddings.
2. Auto-atención causal, con suma residual y normalización.
3. Atención cruzada, con suma residual y normalización.
4. *Feed-forward*, con suma residual y normalización.
5. Dropout.
6. *Softmax* de 5.000.

> ⚠️ Necesitó las mismas dos correcciones que el codificador: los nombres exactos de los atributos y `dropout=rate` en **las dos** atenciones.

In [58]:
# FUNCIÓN CALIFICADA get_DecoderLayer
def get_DecoderLayer(vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate):

    class Decoder(keras.layers.Layer):
        def __init__(self, vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate=0.1):
            super().__init__()
            self.token_emb = keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
            self.pos_emb = keras.layers.Embedding(input_dim=maxlen, output_dim=embed_dim)
            self.self_att = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=rate)
            self.self_layernorm = keras.layers.LayerNormalization()
            self.self_dropout = keras.layers.Dropout(rate)
            self.cross_att = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=rate)
            self.cross_layernorm = keras.layers.LayerNormalization()
            self.cross_dropout = keras.layers.Dropout(rate)
            self.ffn = keras.Sequential([keras.layers.Dense(ff_dim, activation="relu"),
                                         keras.layers.Dense(embed_dim)])
            self.dropout = keras.layers.Dropout(rate)
            self.layernorm = keras.layers.LayerNormalization()
            self.out_dropout = keras.layers.Dropout(rate)
            self.out_layer = keras.layers.Dense(vocab_size, activation="softmax")

        def call(self, inputs, context_tokens):
            positions = keras.ops.arange(start=0, stop=keras.ops.shape(inputs)[-1], step=1)
            x = self.token_emb(inputs) + self.pos_emb(positions)
            attn = self.self_att(x, x, use_causal_mask=True)   # auto-atención causal
            attn = self.self_dropout(attn)
            x = self.self_layernorm(x + attn)
            attn = self.cross_att(x, context_tokens)           # atención cruzada con el encoder
            attn = self.cross_dropout(attn)
            x = self.cross_layernorm(x + attn)
            f = self.ffn(x)
            f = self.dropout(f)
            x = self.layernorm(x + f)
            x = self.out_dropout(x)
            return self.out_layer(x)

    layer = Decoder(vocab_size, maxlen, embed_dim, num_heads, ff_dim, rate)
    return layer

### 📘 Explicación

Pasa por el decodificador los emojis de 10 frases sin su último token, `[:10, :-1]`, junto con la salida del codificador.

In [59]:
# TEST_CELL
decoder_layer = get_DecoderLayer(5000, 25, 128, 2, 32, .1)
decoded_x = decoder_layer(train_emoji_vec[:10,:-1],encoded_x)
print(f'Decodificación de sequencias de entrenamiento: {decoded_x.shape}')

Decodificación de sequencias de entrenamiento: (10, 25, 5000)


#### 🔎 Lectura de la salida

Forma `(10, 25, 5000)`: para cada frase y cada una de sus 25 posiciones, una distribución de probabilidad sobre los 5.000 tokens posibles.

## **Evaluar código**

### 📘 Explicación

**Pruebas del punto 4.** Estructura y valores, como en el punto 3.

In [60]:
grader.run_test("Test 4_1", globals())

Test 4_1


In [61]:
grader.run_test("Test 4_2", globals())

Test 4_2


# **5. Construir modelo**
----
Implemente la función `get_model` que construye el modelo Transformer completo combinando el codificador y decodificador. El modelo debe realizar la tarea de secuencia-a-secuencia para convertir texto en inglés a representaciones con emojis.

**Entrada**:

* **`max_len`**: `int`, Longitud máxima de las secuencias
* **`vocab_size`**: `int`, Tamaño del vocabulario (debe ser el mismo para ambos vocabularios)
* **`embed_dim`**: `int`, Dimensión del espacio de embedding para cada token
* **`num_heads`**: `int`, Número de cabezas de atención
* **`ff_dim`**: `int`, Dimensión de la capa feed-forward
* **`rate`**: `float`, Tasa de dropout para regularización

**Salida**:
* **`model`**: `keras.Model`, Modelo Transformer encoder-decoder completo

> **Nota**: El modelo implementa la arquitectura encoder-decoder típica para tareas de traducción/generación de secuencias, donde el decodificador utiliza tanto su entrada desplazada como el contexto del codificador.

### 📘 Explicación

**Punto 5.** Une codificador y decodificador en un `keras.Model`.

- Las semillas y `clear_session()` venían dadas: reinician Keras para que el modelo salga idéntico en cada llamada.
- `keras.Input(shape=(26,))` es el inglés y `keras.Input(shape=(25,))`, los emojis desplazados. El tamaño del lote no se declara y aparece como `None`.
- `context = enc(encoder_inputs)` pasa el inglés por el codificador.
- `outputs = dec(decoder_inputs, context)` pasa los emojis y el contexto por el decodificador.
- `keras.Model([entradas], salida)` registra el grafo completo con la **API funcional**. El orden de la lista de entradas es el que luego espera `fit`.

In [62]:
# FUNCIÓN CALIFICADA get_model
def get_model(max_len, vocab_size, embed_dim, num_heads, ff_dim, rate):
    # Fijamos una semilla para efectos de reproducibiidad
    np.random.seed(0)
    keras.utils.set_random_seed(0)

    keras.backend.clear_session()

    encoder_inputs = keras.Input(shape=(max_len + 1,))
    decoder_inputs = keras.Input(shape=(max_len,))
    enc = get_EncoderLayer(vocab_size, max_len, embed_dim, num_heads, ff_dim, rate)
    dec = get_DecoderLayer(vocab_size, max_len, embed_dim, num_heads, ff_dim, rate)
    context = enc(encoder_inputs)
    outputs = dec(decoder_inputs, context)
    model = keras.Model([encoder_inputs, decoder_inputs], outputs)
    return model

### 📘 Explicación

Construye el modelo con los hiperparámetros de la prueba e imprime su resumen.

In [63]:
# TEST_CELL
max_len = 25
vocab_size = 5000
embed_dim = 128
num_heads = 2
ff_dim = 32
rate = 0.01

model = get_model(max_len, vocab_size, embed_dim, num_heads, ff_dim, rate)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 26)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder (encoder)   │ (None, 26, 128)   │    784,160 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder (Decoder)   │ (None, 25, 5000)  │  1,561,256 │ input_layer_1[0]… │
│                     │                   │            │ encoder[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,345,416 (8.95 MB)

 Trainable params: 2,345,416 (8.95 MB)

 Non-trainable params: 0 (0.00 B)

#### 🔎 Lectura de la salida

**Coincide exactamente con la salida esperada: 2.345.416 parámetros.**

| Componente | Codificador | Decodificador |
|---|---:|---:|
| `token_emb`, 5.000 × 128 | 640.000 | 640.000 |
| `pos_emb` | 3.328, 26 × 128 | 3.200, 25 × 128 |
| Atención: Q, K, V y salida | 131.968 | 263.936, son dos |
| `ffn` | 8.352 | 8.352 |
| Normalizaciones | 512, dos | 768, tres |
| `out_layer`, 128 × 5.000 + 5.000 | — | 645.000 |
| **Total** | **784.160** | **1.561.256** |

Más de la mitad del modelo está en los dos embeddings de tokens y en la capa de salida, todos atados al vocabulario de 5.000.

**Salida esperada:**
```
Model: "functional_2"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 26)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder (encoder)   │ (None, 26, 128)   │    784,160 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder (Decoder)   │ (None, 25, 5000)  │  1,561,256 │ input_layer_1[0]… │
│                     │                   │            │ encoder[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘
 Total params: 2,345,416 (8.95 MB)
 Trainable params: 2,345,416 (8.95 MB)
 Non-trainable params: 0 (0.00 B)
```

## **Evaluar código**

### 📘 Explicación

**Pruebas del punto 5.** Las dos dieron 100 en el primer intento.

In [64]:
grader.run_test("Test 5_1", globals())

Test 5_1


In [65]:
grader.run_test("Test 5_2", globals())

Test 5_2


# **6. Entrenar el modelo**
----
Complete la función `train_model` que recibe un modelo y lo ajusta usando las secuencias de entrenamiento.

**Entrada**:

* **`model`**: `keras.Model`, modelo de _Keras_ compilado.
* **`train_text`**:  `array`, Arreglo con los textos vectorizados de entrenamiento en inglés.
* **`train_emoji`**:  `array`, Arreglo con los textos vectorizados de entrenamiento de emojis.
* **`test_text`**:  `array`, Arreglo con los textos vectorizados de prueba en inglés.
* **`test_emoji`**:  `array`, Arreglo con los textos vectorizados de prueba en emojis.
* **`learning_rate`**: `float`, tasa de aprendizaje.
* **`epochs`**: `int`, Número de epocas.
* **`batch_size`**: `int`, tamaño del lote.

**Salida**:
* **`history`**: Objeto, objeto que contiene las métricas de evaluación y entrenamiento a lo largo de las épocas.

> **Nota**: Recuerde desfasar por un token la salida esperada de la entrada.



### 📘 Explicación

**Función de pérdida, dada por la plantilla.** Es una entropía cruzada que **ignora el relleno**.

1. `masc` vale 1 donde hay un token real y 0 donde hay relleno.
2. `one_hot(y_true, 5000)` convierte cada id en un vector de 5.000 con un 1 en su posición.
3. `CategoricalCrossentropy(from_logits=False, reduction='none')` mide el error **por posición**. `from_logits=False` indica que la salida ya es *softmax*. `reduction='none'` evita que Keras promedie antes de aplicar la máscara.
4. `loss * masc` anula el relleno, y la división final promedia solo sobre los tokens reales.

En esta tarea cada fila tiene exactamente **dos posiciones que cuentan**: la cadena de emojis y el `[EOS]`. Sin la máscara, el modelo dedicaría su esfuerzo a predecir ceros.

In [66]:
def causal_loss(y_true,y_pred):
    masc = keras.ops.cast(y_true != 0, dtype="float32")
    y_true = keras.ops.one_hot(y_true, 5000, axis=-1)

    loss_fn = keras.losses.CategoricalCrossentropy(from_logits=False, reduction='none')
    loss = loss_fn(y_true, y_pred)
    loss = loss * masc
    loss = keras.ops.sum(loss)/keras.ops.sum(masc)
    return loss

### 📘 Explicación

**Punto 6.** Compila y entrena con el desfase de un token.

- `compile` fija el optimizador Adam, la pérdida `causal_loss` y la métrica `accuracy`. El enunciado dice que el modelo llega compilado, pero `get_model` no lo compila: hay que hacerlo aquí.
- `train_emoji[:, :-1]` quita la última columna y es la **entrada** del decodificador.
- `train_emoji[:, 1:]` quita la primera y es el **objetivo**.
- `validation_data` aplica el mismo desfase a la prueba.

**El desfase para `[BOS] ⚖️ ❓ [EOS]`**

| Posición | Entrada al decodificador | Objetivo |
|---|---|---|
| 0 | `[BOS]` | ⚖️ |
| 1 | ⚖️ | ❓ |
| 2 | ❓ | `[EOS]` |

Gracias a la máscara causal, las 25 predicciones se entrenan a la vez, en una sola pasada.

In [67]:
# FUNCIÓN CALIFICADA train_model
def train_model (model, train_text, train_emoji, test_text, test_emoji, learning_rate, epochs, batch_size):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss=causal_loss, metrics=['accuracy'])
    # El decodificador recibe la secuencia sin el último token
    # y debe predecir la secuencia desfasada un token
    history = model.fit(
        [train_text, train_emoji[:, :-1]], train_emoji[:, 1:],
        validation_data=([test_text, test_emoji[:, :-1]], test_emoji[:, 1:]),
        epochs=epochs, batch_size=batch_size)

    return history

### 📘 Explicación

Entrena 40 épocas con tasa de aprendizaje 0,0001 y lotes de 256: 6.605 / 256 da 26 lotes por época.

In [68]:
history = train_model(model, train_en_vec, train_emoji_vec, test_en_vec, test_emoji_vec, 1e-4, 40, 256)

Epoch 1/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 38s 755ms/step - accuracy: 0.0256 - loss: 8.0254 - val_accuracy: 0.0400 - val_loss: 7.2922
Epoch 2/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.0400 - loss: 7.2663 - val_accuracy: 0.0400 - val_loss: 6.6983
Epoch 3/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.0400 - loss: 6.8950 - val_accuracy: 0.0400 - val_loss: 6.2723
Epoch 4/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.0400 - loss: 6.6223 - val_accuracy: 0.0400 - val_loss: 5.8834
Epoch 5/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.0400 - loss: 6.3719 - val_accuracy: 0.0400 - val_loss: 5.5077
Epoch 6/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.0400 - loss: 6.1313 - val_accuracy: 0.0400 - val_loss: 5.1421
Epoch 7/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.0400 - loss: 5.8985 - val_accuracy: 0.0400 - val_loss: 4.7895
Epoch 8/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.0400 - loss: 5.6773 - val_accuracy: 0.0400 -

#### 🔎 Lectura de la salida

| Época | Pérdida | Accuracy | Pérdida de validación | Accuracy de validación |
|---:|---:|---:|---:|---:|
| 1 | 8,03 | 0,026 | 7,29 | 0,040 |
| 10 | 5,29 | 0,040 | 3,89 | 0,040 |
| **19** | 4,30 | 0,040 | **3,11** | 0,053 |
| 26 | 4,02 | 0,132 | 3,76 | 0,086 |
| 40 | 3,48 | 0,050 | 3,85 | 0,047 |

- **El modelo aprendió.** La pérdida empezó en 8,03, casi ln 5.000 ≈ 8,5, que es lo que da adivinar al azar entre 5.000 tokens, y terminó en 3,48.
- **Sobreajuste desde la época 19.** La validación bajó hasta 3,11 y luego subió hasta cerca de 3,8 mientras el entrenamiento seguía bajando. El mejor modelo habría sido el de la época 19, con *early stopping*.
- **Por qué la accuracy se queda en 0,04.** Es exactamente 1/25. La métrica cuenta las 25 posiciones, incluido el relleno, porque a diferencia de la pérdida **no tiene máscara**. El modelo acierta una: el `[EOS]` de la posición 1, que es trivial. El salto hasta 0,13 hacia la época 26 probablemente viene de posiciones de relleno donde, por azar, empezó a predecir 0; no es una mejora real. Aquí la métrica que informa es la **pérdida de validación**.
- **La validación empieza por debajo del entrenamiento.** La pérdida de entrenamiento se promedia mientras el modelo mejora dentro de la época y con dropout activo; la de validación se mide al final y sin dropout.
- **Comparación con el enunciado.** La salida esperada termina con pérdida de 0,12 y validación de 4,86: memoriza mucho más y generaliza peor. Su accuracy de 0,0805 es casi 2/25: acierta las dos posiciones reales de cada fila de entrenamiento.

**Salida esperada**
```
Epoch 39/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.0804 - loss: 0.1294 - val_accuracy: 0.0411 - val_loss: 4.8453
Epoch 40/40
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.0805 - loss: 0.1232 - val_accuracy: 0.0409 - val_loss: 4.8631
```

veamos algunos ejemplos:

### 📘 Explicación

**Generación de ejemplo, dada por la plantilla.** Es la **decodificación autorregresiva**.

1. Arranca con el token 3, `[BOS]`, en un arreglo de 1 × 25 lleno de ceros.
2. En cada vuelta pasa al modelo la frase de **prueba** `test_en_vec[8]`, que nunca vio, junto con los emojis escritos hasta ahora.
3. Lee la posición *i* de la salida y elige el token más probable con `argmax`: es la decodificación **voraz**.
4. Lo escribe en la posición *i + 1*.
5. Se detiene si el token es 2, `[EOS]`.

Por la máscara causal, la posición *i* solo depende de lo escrito hasta *i*, así que los ceros de la derecha no influyen. Al final, `get_vocabulary()` traduce los ids de vuelta a palabras y emojis.

In [69]:
current_token = 3
text_encoding = np.zeros([1,25])
text_encoding[0,0] = current_token

for i in range(24):
    token_prediction = model([np.expand_dims(test_en_vec[8], axis=0), text_encoding])
    current_token = np.argmax(token_prediction[0,i])
    text_encoding[0,i+1] = int(current_token)
    if current_token == 2:
        break

emoji_vocabulary = np.array(emoji_vectorizer.get_vocabulary())
en_vocabulary = np.array(en_vectorizer.get_vocabulary())
print(' '.join(en_vocabulary[list(test_en_vec[8].astype(int))]))
print(' '.join(emoji_vocabulary[list(text_encoding[0].astype(int))]))

[BOS] we also have a common responsibility to ensure that this takes place with legal certainty while [UNK] fundamental rights [EOS]     
[BOS] 🇪🇺❌🚫 [EOS]                      


#### 🔎 Lectura de la salida

El modelo generó **🇪🇺❌🚫** y luego `[EOS]`.

- **En lo mecánico funcionó:** eligió un token válido del vocabulario de emojis y supo terminar la secuencia.
- **En el significado es parcial:** 🇪🇺 encaja con el dominio del corpus, pero ❌🚫 transmite rechazo y no refleja «responsabilidad común» ni «derechos fundamentales».

Es coherente con una pérdida de validación cercana a 3,8: el modelo captó el tema general del corpus, no el matiz de cada frase. El `[UNK]` del inglés es una palabra que no estaba entre las 5.000 más frecuentes.

## **Evaluar código**

### 📘 Explicación

**Prueba del punto 6.** Califica la función `train_model`, no los números del registro, que dependen del hardware y de la versión de Keras.

In [70]:
grader.run_test("Test 6_1", globals())

Test 6_1
26/26 ━━━━━━━━━━━━━━━━━━━━ 33s 615ms/step - accuracy: 0.0256 - loss: 8.0254 - val_accuracy: 0.0400 - val_loss: 7.2922
26/26 ━━━━━━━━━━━━━━━━━━━━ 33s 631ms/step - accuracy: 0.0256 - loss: 8.0254 - val_accuracy: 0.0400 - val_loss: 7.2922


# **Evaluación**

### 📘 Explicación

**Envío oficial.** Vuelve a ejecutar las 11 pruebas en el servidor y registra la entrega en UNCode.

In [71]:
grader.submit_task(globals())

Test 1_1
Test 1_2
Test 2_1
Test 2_2
Test 3_1
Test 3_2
Test 4_1
Test 4_2
Test 5_1
Test 5_2
Test 6_1
26/26 ━━━━━━━━━━━━━━━━━━━━ 33s 622ms/step - accuracy: 0.0256 - loss: 8.0254 - val_accuracy: 0.0400 - val_loss: 7.2922
26/26 ━━━━━━━━━━━━━━━━━━━━ 32s 618ms/step - accuracy: 0.0256 - loss: 8.0254 - val_accuracy: 0.0400 - val_loss: 7.2922


#### 🔎 Lectura de la salida

**Resultado: las 11 pruebas en 100 y «Your submission grade: 100.0%».**

Revisa en UNCode tus submissions en la [página de la tarea.](https://uncode.unal.edu.co/course/DL-GroupMLDS-5-2026-2/8c487bd7-4c95-4b75-88d5-9e7093aae729)

# 📘 Cierre

| Punto | Función | Primera versión | Versión final | Qué cambió |
|---|---|---|---|---|
| 1 | `preprocess` | 80, 80 | 100, 100 | Emojis sin separar |
| 2 | `vectorize` | 100, 100 | 100, 100 | — |
| 3 | `get_EncoderLayer` | 0, 10 | 100, 100 | Nombres de atributos y `dropout=rate` |
| 4 | `get_DecoderLayer` | 0, 100 | 100, 100 | Nombres de atributos y `dropout=rate` |
| 5 | `get_model` | 100, 100 | 100, 100 | — |
| 6 | `train_model` | 100 | 100 | — |

**Lo esencial para recordar**
1. **Codificador y decodificador tienen trabajos distintos:** uno lee y otro escribe. Se comunican solo por la **atención cruzada**.
2. **La máscara causal** hace que el decodificador solo mire hacia atrás; sin ella no podría generar.
3. **Teacher forcing:** entrada `[:, :-1]` y objetivo `[:, 1:]`. De ahí salen las longitudes 26 y 25.
4. **La pérdida enmascarada** ignora el relleno; la accuracy sin máscara, no, y por eso engaña aquí.
5. **Un calificador automático también evalúa la forma del código:** nombres, tipos y parámetros por defecto.
6. **Ante una contradicción entre la instrucción escrita y un ejemplo impreso, manda la instrucción.**

**Mejoras posibles, fuera de la calificación**
- *Early stopping* en la época de mejor validación.
- Separar los emojis por grafemas con el paquete `regex` y `\X`, para que ✅ sea un token compartido.
- Una accuracy con máscara.
- Más bloques o búsqueda en haz en lugar de `argmax`.


# **Créditos**
---

* **Profesor:** [Fabio Augusto Gonzalez](https://dis.unal.edu.co/~fgonza/)
* **Asistentes docentes :**
  * [Juan Sebastián Malagón Torres](https://http://juselara.com/)
* **Coordinador de virtualización:**
    - [Edder Hernández Forero](https://www.linkedin.com/in/edder-hernandez-forero-28aa8b207/).

**Universidad Nacional de Colombia** - *Facultad de Ingeniería*